# Jour 1a — Introduction à la biologie et préparation des données

## ADN codant vs. non-codant

Un génome bactérien est une seule et longue chaîne d'ADN. Une partie seulement de cette
chaîne **code réellement pour une protéine** (une *CDS*, séquence codante — lue par
triplets appelés codons, chacun spécifiant un acide aminé). Le reste est **non-codant** :
régions intergéniques, séquences régulatrices, etc.

Notre tâche cette semaine : **étant donné une courte fenêtre d'ADN, prédire si elle se
trouve à l'intérieur d'une CDS ou non.**

C'est un problème réel et utile — la recherche de gènes (*gene finding*) est une tâche
fondamentale en bioinformatique, et elle est binaire, ce qui la rend abordable pour un
premier projet.

## Regardons d'abord les fichiers bruts

Pas de prétraitement pour l'instant. Chaque génome dans `data/supervised/raw/` est une
paire de fichiers :
- un fichier **FASTA** : la séquence brute du génome
- un fichier **GFF** : les annotations — où se trouvent les gènes/régions CDS, sur quel brin

Regardons les deux directement.

In [ ]:
import sys
sys.path.append("src")

RAW_DIR = "../../data/supervised/raw"

In [ ]:
# TODO : ouvrez le fichier FASTA ci-dessous et affichez ses 6 premières lignes
# indice : with open(...) as f: for _ in range(6): print(f.readline().rstrip())
fasta_path = f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.fasta"
with open(fasta_path) as f:
    for _ in range(6):
        print(...)  # TODO : remplacez ... par la ligne à afficher

In [ ]:
# TODO : lisez le fichier GFF, ignorez les lignes de commentaire (qui commencent par '#'),
# puis affichez les 8 premières lignes restantes. Repérez les lignes de type CDS
# (colonnes : seqid, source, type, start, end, score, strand, phase, attributes).
gff_path = f"{RAW_DIR}/train/GCA_000240015.1_ASM24001v1.gff"
with open(gff_path) as f:
    lines = ...  # TODO : gardez uniquement les lignes qui ne commencent pas par '#'
for l in lines[:8]:
    print(l.rstrip())

## Des fichiers bruts à un jeu de données étiqueté

`src/build_dataset.py` est le script fourni qui transforme cela en quelque chose sur lequel
on peut entraîner un modèle. Pour chaque génome, il :

1. Lit tous les intervalles CDS du GFF (`type == "CDS"`, colonnes `start`/`end`/`strand`).
2. Fait glisser des fenêtres de longueur fixe (200 pb) **à l'intérieur** des régions CDS →
   étiquette `1` (codant). Si la CDS est sur le brin `-`, la fenêtre est d'abord retournée
   en complément inverse, pour que le modèle voie toujours la séquence dans l'orientation
   codante.
3. Calcule les **intervalles** entre les régions CDS (tout ce qui n'est couvert par aucune
   CDS) et fait glisser les mêmes fenêtres dessus → étiquette `0` (non-codant).
4. Équilibre les classes par génome, mélange, et écrit un CSV par split.

Il a déjà été exécuté une fois (`data/supervised/processed/{train,val,test}.csv`
existent). Si vous voulez changer la taille de fenêtre ou le ré-échantillonnage,
relancez-le :

```bash
python ../../code/src/build_dataset.py --raw_dir ../../data/supervised/raw \
    --out_dir ../../data/supervised/processed --window 200 --stride 200
```

In [ ]:
# TODO : chargez les trois splits avec load_all(...), puis affichez pour chacun
# sa forme (shape) et l'équilibre des classes (value_counts sur la colonne 'label')
from data import load_all

splits = ...
for name, df in splits.items():
    print(name, ..., "class balance:", ...)

splits["train"].head()

### Point de contrôle

Vous devriez avoir des DataFrames `train`, `val`, `test`, chacun avec une colonne
`sequence` (la fenêtre brute de 200 pb) et une colonne `label` (1 = codant, 0 = non-codant),
à peu près équilibrées entre les classes.

Suite : `01_kmer_and_cnn_baselines.ipynb` — construire les premiers modèles à partir de ces
données.

*Bloqué ? La version complète est dans `solution/00_biology_intro_and_data_setup.ipynb`.*